In [1]:
import numpy as np


class DecisionStump:

  def __init__(self):
    self.polarity = 1
    self.feature_idx = None
    self.threshold = None
    self.alpha = None

  def predict(self, X):
    n_samples = X.shape[0]
    X_column = X[:, self.feature_idx]
    predictions = np.ones(n_samples)

    if self.polarity == 1:
      predictions[X_column < self.threshold] = -1
    else:
      predictions[X_column > self.threshold] = -1

    return predictions


class AdaBoostClassifierScratch:

  def __init__(self, n_estimators=10):
    self.n_estimators = n_estimators
    self.estimators = []

  def fit(self, X, y):
    n_samples, n_features = X.shape

    # 1. Initialize sample weights uniformly: w_i = 1 / N
    w = np.full(n_samples, (1.0 / n_samples))

    self.estimators = []

    for _ in range(self.n_estimators):
      stump = DecisionStump()
      min_error = float("inf")

      # 2. Greedy search for the best feature, threshold, and polarity
      for feature_i in range(n_features):
        X_column = X[:, feature_i]
        thresholds = np.unique(X_column)

        for threshold in thresholds:
          for polarity in [1, -1]:
            # Generate predictions
            predictions = np.ones(n_samples)
            if polarity == 1:
              predictions[X_column < threshold] = -1
            else:
              predictions[X_column > threshold] = -1

            # Compute weighted classification error: epsilon_t = sum(w_i * (y_i != h(x_i)))
            misclassified = w[y != predictions]
            error = np.sum(misclassified)

            if error < min_error:
              min_error = error
              stump.polarity = polarity
              stump.threshold = threshold
              stump.feature_idx = feature_i

      # Numerical stability safeguard against division by zero
      EPS = 1e-10
      min_error = np.clip(min_error, EPS, 1.0 - EPS)

      # 3. Calculate learner voting weight: alpha_t = 0.5 * ln((1 - error) / error)
      stump.alpha = 0.5 * np.log((1.0 - min_error) / min_error)

      # 4. Update sample weights: w_i = w_i * exp(-alpha * y_i * h(x_i))
      predictions = stump.predict(X)
      w *= np.exp(-stump.alpha * y * predictions)

      # 5. Normalize sample weights: sum(w_i) = 1
      w /= np.sum(w)

      # Save current weak learner
      self.estimators.append(stump)

  def predict(self, X):
    # Final ensemble decision: sign( sum( alpha_t * h_t(x) ) )
    stump_preds = [stump.alpha * stump.predict(X) for stump in self.estimators]
    y_pred = np.sum(stump_preds, axis=0)
    return np.sign(y_pred)


# =========================================================================
# DEMO EXECUTION
# =========================================================================
if __name__ == "__main__":
  np.random.seed(42)

  # Synthetic Binary Classification Dataset: [Feature 1, Feature 2]
  # Note: Labels must be -1 and +1 for standard AdaBoost mathematics
  X = np.array([
      [1.0, 2.1],
      [1.5, 1.8],
      [2.0, 2.5],
      [3.0, 3.2],
      [3.5, 2.9],
      [4.0, 4.1],
      [4.5, 3.8],
      [5.0, 5.2],
  ])
  y = np.array([-1, -1, -1, -1, 1, 1, 1, 1])

  # Train AdaBoost with 5 sequential decision stumps
  adaboost = AdaBoostClassifierScratch(n_estimators=5)
  adaboost.fit(X, y)

  # Evaluate on training data
  predictions = adaboost.predict(X)
  accuracy = np.mean(predictions == y) * 100

  print("=" * 60)
  print("  ADABOOST CLASSIFIER FROM SCRATCH (NUMPY)")
  print("=" * 60)
  print(f"Number of Sequential Stumps : {len(adaboost.estimators)}")
  print(f"Ground Truth Labels         : {y}")
  print(f"Model Predictions           : {predictions.astype(int)}")
  print(f"Training Classification Acc : {accuracy:.2f}%")
  print("-" * 60)
  print("Learner Voting Weights (alpha):")
  for idx, s in enumerate(adaboost.estimators):
    print(
        f"  Stump {idx+1}: Feature {s.feature_idx} | Threshold: {s.threshold:.2f} | Alpha: {s.alpha:.4f}"
    )
  print("=" * 60)

  ADABOOST CLASSIFIER FROM SCRATCH (NUMPY)
Number of Sequential Stumps : 5
Ground Truth Labels         : [-1 -1 -1 -1  1  1  1  1]
Model Predictions           : [-1 -1 -1 -1  1  1  1  1]
Training Classification Acc : 100.00%
------------------------------------------------------------
Learner Voting Weights (alpha):
  Stump 1: Feature 0 | Threshold: 3.50 | Alpha: 11.5129
  Stump 2: Feature 0 | Threshold: 3.50 | Alpha: 11.5129
  Stump 3: Feature 0 | Threshold: 3.50 | Alpha: 11.5129
  Stump 4: Feature 0 | Threshold: 3.50 | Alpha: 11.5129
  Stump 5: Feature 0 | Threshold: 3.50 | Alpha: 11.5129


In [ ]:
#To load and train the model on your own dataset (such as a CSV file), ensure two key conditions are met:Numeric Encoding: All feature columns in X must be numerical (convert categorical variables with One-Hot or Label Encoding).Binary Labels: AdaBoost mathematically expects binary targets to be -1 and +1 (convert standard 0/1 labels to -1/+1).
#1. Numeric Encoding: All feature columns in X must be numerical (convert categorical variables with One-Hot or Label Encoding).
#2.Binary Labels: AdaBoost mathematically expects binary targets to be -1 and +1 (convert standard 0/1 labels to -1/+1).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# 1. Load your CSV file
df = pd.read_csv("your_dataset.csv")

# 2. Separate features (X) and target label (y)
# Replace 'target_column_name' with your actual label column
X = df.drop(columns=["target_column_name"]).values
y = df["target_column_name"].values

# 3. CRITICAL: Convert labels from {0, 1} to {-1, +1}
# If your dataset labels are already -1 and 1, you can skip this line
y = np.where(y == 0, -1, 1)

# 4. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Fit the custom AdaBoost model
model = AdaBoostClassifierScratch(n_estimators=50)
model.fit(X_train, y_train)

# 6. Generate Predictions & Evaluate
y_pred = model.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Class -1", "Class +1"]))